# Temporal Analysis and Aggregation

Use this notebook after temporal inference jobs complete.

It does three things:
1. Merges per-sample outputs from `results/*_temporal_parts/*.pkl`
2. Exports consolidated predictions to `results/temporal_pred.csv`
3. Computes temporal score summaries (Gaussian scoring)

Expected input:
- `../../data/labels.csv`
- `results/molmo_temporal_parts/*.pkl`
- `results/qwen_temporal_parts/*.pkl`

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from dataset.accident_dataset import get_dataset_paths, resolve_dataset_root


def get_scores(df, col_pred):
    scores = []
    for _, row in df.iterrows():
        time_gt = row['accident_time']
        time_pred = row[col_pred]
    
        if time_pred is None:
            scores.append({
                'sigma-0.5': 0,
                'sigma-1.0': 0,
                'sigma-2.0': 0,
            })

        else:
            scores.append({
                'sigma-0.5': np.exp(-(time_pred - time_gt)**2 / (2*(0.5**2)) ),
                'sigma-1.0': np.exp(-(time_pred - time_gt)**2 / (2*(1.0**2)) ),
                'sigma-2.0': np.exp(-(time_pred - time_gt)**2 / (2*(2.0**2)) ),
            })
    
    return pd.DataFrame(scores)

# Run the temporal reasoning
- Note: This takes lot of time (12+ hours). Use main.py with --range parameter to split it into viable chunks for easier paralellization.
- Example: This will run first 1/3 of the dataset: `main.py --model qwen --range 0:676`

In [ ]:
# main.py --model qwen --range 0:676
# main.py --model qwen --range 676:1352
# main.py --model qwen --range 1352:2027

# main.py --model molmo --range 0:676
# main.py --model molmo --range 676:1352
# main.py --model molmo --range 1352:2027

# Collect results
- By default, results one file per video is stored.
- This section collects the results into single dataframe, which will be used for end-to-end results

In [ ]:
# Process partial results into single dataframe
dataset_path = Path('../../dataset')
dataset_root = resolve_dataset_root(dataset_path)
_, metadata_path = get_dataset_paths(
    dataset_root, kind="real"
)
df = pd.read_csv(metadata_path)
df["video_path"] = df["path"].apply(lambda x: os.path.join(dataset_root, x))

baseline_dir = '.'
results_molmo = {}
results_qwen = {}
for i, row in df.iterrows():
    data = torch.load(f'{baseline_dir}/results/molmo_temporal_parts/{i}.pkl', weights_only=False)
    results_molmo[i] = data['temporal']

    data = torch.load(f'{baseline_dir}/results/qwen_temporal_parts/{i}.pkl', weights_only=False)
    results_qwen[i] = data['temporal']


pred_molmo = pd.DataFrame(results_molmo).T
df['pred_frame_molmo'] = pred_molmo['pred_frame'].fillna(df['no_frames'] // 2)
df['pred_ts_molmo'] = pred_molmo['pred_ts'].fillna(df['accident_time'] // 2)

pred_qwen = pd.DataFrame(results_qwen).T
df['pred_frame_qwen'] = pred_qwen['pred_frame'].fillna(df['no_frames'] // 2)
df['pred_ts_qwen'] = pred_qwen['pred_ts'].fillna(df['accident_time'] // 2)

df.to_csv('results/temporal_pred.csv')

# Calculate scores

In [ ]:
# Molmo temporal predictions
get_scores(df, 'pred_ts_molmo').mean()

In [ ]:
# Qwen temporal predictions
get_scores(df, 'pred_ts_qwen').mean()